In this tutorial, let us perform an operation over a vector of fixed dimension and a fixed workgroup size. In this case, the total number of work items needs to be chosen accordingly.

In [ ]:
!pip install pyopencl

In [ ]:
import numpy as np
import pyopencl as cl
import pyopencl.array as cl_array

In [ ]:
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]
print("Platform name:", platform.name)
print("Device name:", device.name)
print("Maximum work group size:", device.max_work_group_size)

In [ ]:
ctx = cl.create_some_context()
queue = cl.CommandQueue(ctx, properties=cl.command_queue_properties.PROFILING_ENABLE)

Let us create a vector of fixed dimension, with increasing values from zero.

In [ ]:
dim = 100
np_vector = np.arange(dim, dtype=np.int32)
print("Input vector:", np_vector)
cl_vector = cl_array.to_device(queue, np_vector)

The kernel function doubles the value of the array.

In [ ]:
kernel = """
__kernel void double_nocheck(
    __global const int* a,
    __global int* b
)
{
    int id = get_global_id(0);
    b[id] = a[id] * 2;
}
__kernel void double_withcheck(
    __global const int* a,
    __global int* b,
    const int n
)
{
    int id = get_global_id(0);
    if (id < n) {
        b[id] = a[id] * 2;
    }
}
"""

In [ ]:
prg = cl.Program(ctx, kernel).build()

Let us choose a fixed size of the work group.

In [ ]:
local_work_group_size = 8

In [ ]:
print("Size of vector:", dim)
print("Local work group size:", local_work_group_size)
print("Number of full groups:", dim // local_work_group_size)
print("Remaining work items:", dim % local_work_group_size)

The global size of the work items needs to be a multiple of the local group size, to have groups that have equal number of threads. To cover the entire vector, we need to find the first multiple of the work group size that is larger than the dimension.

In [ ]:
def next_multiple(val, mul):
    """Return the smallest value which is larger or equal to 'val' and a multiple of 'mul'."""
    return -(-val // mul) * mul

In [ ]:
global_work_group_size = next_multiple(dim, local_work_group_size)
print("Global work group size:", global_work_group_size)

This means that in many cases there will be *dummy* threads present at the end. These are threads that are not performing any useful calculations. Generally speaking, there are two solutions:


1.   Create an input vector with the same size of the work group. The final threads calculate the results but the host will never use them. This is also called *padding*.
2.   Adapt the kernel, so that only the threads with global identifier smaller than the problem dimension perform the calculation.



In [ ]:
np_vector_padded = np.pad(np_vector, (0, global_work_group_size-dim))
print("Padded input vector:", np_vector_padded)
cl_vector_padded = cl_array.to_device(queue, np_vector_padded)

In [ ]:
# Version 1: unsafe calculation
cl_output_1 = cl_array.zeros_like(cl_vector)
event = prg.double_nocheck(
    queue,
    (global_work_group_size,), (local_work_group_size,),
    cl_vector.data, cl_output_1.data,
)
print("Output 1:", cl_output_1)

# Version 2: safe calculation with padded vector
cl_output_2 = cl_array.zeros_like(cl_vector_padded)
event = prg.double_nocheck(
    queue,
    (global_work_group_size,), (local_work_group_size,),
    cl_vector.data, cl_output_2.data,
)
print("Output 2:", cl_output_2)

# Version 3: safe calculation with check in kernel
cl_output_3 = cl_array.zeros_like(cl_vector)
event = prg.double_withcheck(
    queue,
    (global_work_group_size,), (local_work_group_size,),
    cl_vector.data, cl_output_3.data, np.int32(dim),
)
print("Output 3:", cl_output_3)

The output of the unsafe implementation is correct. However, the kernel also calculated values of the output vector outside the range of the declared vector. This means that values in the memory were changed, which could have been allocated by other variable or processes.

The output of the padded implementation is correct. However, more memory was required and the output vector should be cut short on the host if this vector is needed later on.

The output of the kernel with a check is correct and the output vector has the correct size. However, the if-statement in the kernel causes computational overhead.